# Optimisation du portefeuille d'actifs financiers Maghrebia

Ce notebook utilise les rendements attendus APT produits par le diagnostic pré-optimisation et compare plusieurs allocations sous contraintes. Les contrôles techniques détaillés sont externalisés dans `scripts/quality_check_notebook_02.py` afin de conserver ici une lecture financière claire.


## 1. Objectif du notebook

Ce notebook transforme les inputs validés du diagnostic en allocations optimisées sous contraintes. Le coeur méthodologique reste l'implémentation interne : rendements attendus APT, covariance APT, contraintes internes, contraintes réglementaires testables, Mean-CVaR et scoring multicritère.

Markowitz est conservé comme benchmark académique historique afin d'illustrer le compromis rendement-risque classique. Les briques complémentaires ne forment pas un catalogue de modèles : elles servent à tester la robustesse de la recommandation face au risque extrême, au turnover, à la concentration, à l'incertitude de distribution et aux contraintes institutionnelles d'une compagnie d'assurances.

Les benchmarks skfolio sont utilisés uniquement comme contrôle externe de cohérence. Ils ne remplacent pas le moteur interne du projet.


In [1]:
from pathlib import Path
from dataclasses import replace
import json
import math
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

try:
    pio.renderers.default = "notebook_connected"
except Exception:
    pio.renderers.default = "iframe_connected"
pio.templates.default = "plotly_white"

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from maghrebia_quant.apt_weekly import APT_OPTIMISTIC_FACTOR_TREATMENT
from maghrebia_quant.optimization_apt import (
    APTOptimizationConfig,
    build_asset_alignment,
    build_context,
    build_multiobjective_scores,
    build_regulatory_constraints_map,
    build_universe,
    evaluate_constraints,
    export_apt_scenario_sheets,
    final_recommendation,
    generate_monte_carlo,
    load_apt_optimization_inputs,
    load_apt_mu_scenarios,
    optimized_weights_table,
    portfolio_metrics,
    references_table,
    risk_contributions,
    run_apt_scenario_analysis,
    solve_all_models,
    solve_efficient_frontier,
    stress_tests_by_portfolio,
)
from maghrebia_quant.skfolio_benchmark import run_skfolio_benchmarks
from maghrebia_quant.notebook02_robustness import (
    bootstrap_stability_check,
    build_factor_risk_sensitivity,
    build_pareto_filter,
    build_robust_cvar_wasserstein,
    build_rorac_proxy,
    solve_mean_cvar_turnover_constrained,
)

EXPORT_DIR = PROJECT_DIR / "exports"
FIGURES_DIR = EXPORT_DIR / "figures" / "notebook_02"
QUALITY_DIR = EXPORT_DIR / "quality"
EXCEL_OUTPUT = EXPORT_DIR / "notebook_02_optimisation_resultats.xlsx"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_DIR.mkdir(parents=True, exist_ok=True)
for old_html in FIGURES_DIR.glob("*.html"):
    old_html.unlink()

def project_relative(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_DIR))
    except Exception:
        return str(path)

def show_and_export_fig(fig, name, output_dir=FIGURES_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.show()
    path = output_dir / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    return path

def pct(x):
    return "" if pd.isna(x) else f"{float(x):.2%}"

N_MONTE_CARLO_REQUESTED = 100_000
N_MONTE_CARLO_EFFECTIVE = 50_000  # fallback reproductible pour garder un temps d'exécution raisonnable

CONFIG = replace(
    APTOptimizationConfig(),
    frontier_points=500,
    slsqp_random_starts=12,
    monte_carlo_required=N_MONTE_CARLO_EFFECTIVE,
    monte_carlo_max_attempts=600_000,
)
np.random.seed(CONFIG.random_seed)


## 2. Chargement des inputs validés du notebook 01

Les données chargées proviennent des exports validés du diagnostic : univers optimisable, rendements historiques, rendements attendus APT, scénarios APT, covariance, contraintes internes et cartographie réglementaire.


In [2]:
data = load_apt_optimization_inputs(PROJECT_DIR)
universe = build_universe(data)
context = build_context(data, universe, PROJECT_DIR)
context["rf_annual"] = float(data["rf_annual"])
mu_APT = data["mu"].reindex(universe["asset_id"])
Sigma_APT = data["sigma"].reindex(index=mu_APT.index, columns=mu_APT.index)
Returns_Historical = data["returns"].reindex(columns=mu_APT.index)
Regulatory_Constraints_Map = build_regulatory_constraints_map(referential_found=False)
Asset_Alignment = build_asset_alignment(data, universe)
Optimization_References = references_table()

dates = pd.to_datetime(Returns_Historical.index)
median_gap = dates.to_series().diff().dt.days.median()
RETURN_FREQUENCY = "daily" if median_gap <= 2 else ("weekly" if median_gap <= 8 else "other")
PERIODS_PER_YEAR = 252 if RETURN_FREQUENCY == "daily" else 52

Hypotheses = pd.DataFrame([
    {"Hypothese": "Taux sans risque", "Valeur": float(data["rf_annual"]), "Commentaire": "Taux court BCT issu du notebook 01."},
    {"Hypothese": "Rendements attendus", "Valeur": "APT central", "Commentaire": "Scénario central utilisé comme référence."},
    {"Hypothese": "Scénarios de sensibilité", "Valeur": "APT prudent / central / optimiste / Historical_Raw", "Commentaire": "Historical_Raw est descriptif et non recommandé comme hypothèse prospective."},
    {"Hypothese": "Covariance", "Valeur": "Sigma_APT", "Commentaire": "Covariance factorielle APT issue du notebook 01."},
    {"Hypothese": "Fréquence des rendements historiques", "Valeur": RETURN_FREQUENCY, "Commentaire": f"{len(Returns_Historical)} observations, annualisation {PERIODS_PER_YEAR}."},
    {"Hypothese": "Monte Carlo", "Valeur": CONFIG.monte_carlo_required, "Commentaire": f"Portefeuilles faisables sous contraintes ; cible demandée {N_MONTE_CARLO_REQUESTED:,}, fallback exécuté {N_MONTE_CARLO_EFFECTIVE:,}."},
    {"Hypothese": "Points frontière efficiente", "Valeur": CONFIG.frontier_points, "Commentaire": "Frontière dense pour lecture rendement-risque."},
])

Inputs = universe[["asset_id", "asset_name", "asset_class", "asset_type", "current_value", "current_weight_optimisable", "expected_return_annualized"]].copy()
Inputs = Inputs.merge(Asset_Alignment[["asset_id", "final_status"]], on="asset_id", how="left")
display(Hypotheses)
display(Inputs.head(10))


,Hypothese,Valeur,Commentaire
0,Taux sans risque,0.076408,Taux court BCT issu du notebook 01.
1,Rendements attendus,APT central,Scénario central utilisé comme référence.
2,Scénarios de sensibilité,APT prudent / central / optimiste / Historical...,Historical_Raw est descriptif et non recommand...
3,Covariance,Sigma_APT,Covariance factorielle APT issue du notebook 01.
4,Fréquence des rendements historiques,daily,"246 observations, annualisation 252."
5,Monte Carlo,50000,Portefeuilles faisables sous contraintes ; cib...
6,Points frontière efficiente,500,Frontière dense pour lecture rendement-risque.


,asset_id,asset_name,asset_class,asset_type,current_value,current_weight_optimisable,expected_return_annualized,final_status
0,BTA_7_4_02_2030,"BTA 7,4% 02/2030",Titres de l'État,government_bond,3.246080e+07,0.098109,0.100974,OK
1,BTA_7_5_12_2028,"BTA 7,5% 12/2028",Titres de l'État,government_bond,2.259674e+07,0.068296,0.100101,OK
2,EMPRUNT_NATIONAL_2021_2EME_TRANCHE,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,government_bond,2.892336e+07,0.087417,0.099397,OK
3,EMPRUNT_NATIONAL_2022_3EME_TRANCHE_10_ANS,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,government_bond,1.450828e+07,0.043850,0.100644,OK
4,EMPRUNT_NATIONAL_2023_4EME_TRANCHE_CATEGORIE_C,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,government_bond,1.643861e+07,0.049684,0.101508,OK
5,TN0007310568,E.O HL 2020-03,Obligations corporate,corporate_bond,3.298941e+06,0.009971,0.109896,OK
6,TN0003100872,EO BNA SUB 2021-1,Obligations corporate,corporate_bond,6.297656e+06,0.019034,0.108020,OK
7,TNHFN9X6ARP3,EO ADVANS 2022-1,Obligations corporate,corporate_bond,9.483725e+06,0.028663,0.112136,OK
8,TN8DSPQCBC06,EO ATL 2022-1,Obligations corporate,corporate_bond,2.790776e+06,0.008435,0.111068,OK
9,TNL7VQZVHR54,EO HL 2023-1,Obligations corporate,corporate_bond,1.378937e+08,0.416768,0.111433,OK


## 3. Hypothèses financières et scénarios de rendements attendus

Le scénario APT central reste la référence de l'optimisation principale. Les scénarios prudent et optimiste servent aux tests de sensibilité. Le scénario Historical_Raw correspond au rendement historique annualisé brut ; il est conservé à titre comparatif, mais ne constitue pas une hypothèse prospective recommandée car il peut surestimer les rendements attendus sur un échantillon court.


In [3]:
current_weights = universe["current_weight_optimisable"].to_numpy(float)
current_weights = current_weights / current_weights.sum()
current_row, Current_Regulatory_Check = portfolio_metrics(
    "Portefeuille actuel",
    current_weights,
    mu_APT,
    Sigma_APT,
    Returns_Historical,
    float(data["rf_annual"]),
    current_weights,
    universe,
    context,
    Regulatory_Constraints_Map,
    "BENCHMARK",
)
Current_Portfolio = pd.DataFrame([current_row])
Current_Portfolio_Display = Current_Portfolio[["portfolio_name", "expected_return_APT", "volatility_APT", "sharpe_ratio", "var_95_historical", "cvar_95_historical", "max_drawdown", "regulatory_status"]]
display(Current_Portfolio_Display)


,portfolio_name,expected_return_APT,volatility_APT,sharpe_ratio,var_95_historical,cvar_95_historical,max_drawdown,regulatory_status
0,Portefeuille actuel,0.106826,0.015454,1.968308,0.000499,0.001023,-0.002807,COMPLIANT_WITH_UNTESTED_CONSTRAINTS


## 4. Diagnostic synthétique du portefeuille actuel

Le Sharpe historique est descriptif. Il doit être interprété avec prudence lorsque les séries obligataires sont revalorisées par modèle, car ce traitement peut lisser la volatilité observée. L'optimisation s'appuie donc principalement sur les rendements attendus APT.


In [4]:
asset_metrics_src = data["Asset_Metrics"].copy()
Asset_Risk_Interpretation = universe[["asset_id", "asset_name", "asset_class", "asset_type"]].merge(
    asset_metrics_src[["asset_id", "annualized_return_geometric", "annualized_volatility", "sharpe_rf_short_term", "loss_var_95", "loss_cvar_95"]],
    on="asset_id",
    how="left",
).merge(
    data["expected"][["asset_id", "mu_apt_raw", "mu_apt_central", "mu_apt_prudent", "mu_apt_optimistic"]],
    on="asset_id",
    how="left",
)
Asset_Risk_Interpretation = Asset_Risk_Interpretation.rename(columns={
    "annualized_return_geometric": "historical_return_annualized",
    "annualized_volatility": "historical_volatility",
    "sharpe_rf_short_term": "sharpe_historical",
    "loss_var_95": "var_95_historical",
    "loss_cvar_95": "cvar_95_historical",
    "mu_apt_central": "expected_return_APT",
})
conditions = [
    Asset_Risk_Interpretation["sharpe_historical"].gt(5),
    Asset_Risk_Interpretation["sharpe_historical"].gt(3),
    Asset_Risk_Interpretation["historical_volatility"].lt(0.01),
]
choices = ["SHORT_SAMPLE_LIMITATION", "HIGH_SHARPE_INTERPRET_WITH_CAUTION", "DCF_SMOOTHING_EFFECT"]
Asset_Risk_Interpretation["Interpretation_Status"] = np.select(conditions, choices, default="OK")
display(Asset_Risk_Interpretation[["asset_name", "asset_class", "historical_return_annualized", "expected_return_APT", "historical_volatility", "sharpe_historical", "Interpretation_Status"]].sort_values("sharpe_historical", ascending=False).head(12))


,asset_name,asset_class,historical_return_annualized,expected_return_APT,historical_volatility,sharpe_historical,Interpretation_Status
7,EO ADVANS 2022-1,Obligations corporate,0.113449,0.112136,0.004988,6.788803,SHORT_SAMPLE_LIMITATION
2,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,0.103354,0.099397,0.004173,5.930114,SHORT_SAMPLE_LIMITATION
5,E.O HL 2020-03,Obligations corporate,0.102877,0.109896,0.004165,5.837282,SHORT_SAMPLE_LIMITATION
9,EO HL 2023-1,Obligations corporate,0.114846,0.111433,0.006191,5.673168,SHORT_SAMPLE_LIMITATION
10,EO BH LEASING 2023-1,Obligations corporate,0.113626,0.111341,0.006018,5.654234,SHORT_SAMPLE_LIMITATION
8,EO ATL 2022-1,Obligations corporate,0.111110,0.111068,0.005680,5.591515,SHORT_SAMPLE_LIMITATION
1,"BTA 7,5% 12/2028",Titres de l'État,0.114466,0.100101,0.007381,4.713222,HIGH_SHARPE_INTERPRET_WITH_CAUTION
3,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,0.112960,0.100644,0.007407,4.513713,HIGH_SHARPE_INTERPRET_WITH_CAUTION
0,"BTA 7,4% 02/2030",Titres de l'État,0.117582,0.100974,0.009326,4.031190,HIGH_SHARPE_INTERPRET_WITH_CAUTION
4,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,0.116344,0.101508,0.009064,4.025149,HIGH_SHARPE_INTERPRET_WITH_CAUTION


## 5. Covariance APT et risque

La matrice de covariance utilisée dans l'optimisation est celle validée dans le notebook 01. Les rendements historiques périodiques restent utilisés pour VaR, CVaR, stress tests, bootstrap et benchmarks externes.


In [5]:
Covariance = Sigma_APT.copy()
Correlation = Returns_Historical.corr()
Covariance_Diagnostics = pd.DataFrame([{
    "Dimension": Covariance.shape[0],
    "Symmetric": bool(np.allclose(Covariance, Covariance.T, atol=1e-10)),
    "Min_Eigenvalue": float(np.linalg.eigvalsh(Covariance.to_numpy(float)).min()),
    "Sigma_Repaired_By_Notebook01": bool(data.get("sigma_repaired", False)),
}])
display(Covariance_Diagnostics)


,Dimension,Symmetric,Min_Eigenvalue,Sigma_Repaired_By_Notebook01
0,21,True,0.000004,False


## 6. Contraintes internes et réglementaires

Les contraintes réglementaires testables sont vérifiées avec le moteur interne. Lorsqu'une contrainte n'est pas testable faute de données détaillées, elle reste documentée sans être présentée comme conforme.


In [6]:
Constraints = Regulatory_Constraints_Map.copy()
constraint_cols = [
    "constraint_id", "constraint_name", "threshold_type", "threshold_value",
    "denominator", "applies_to", "implementation_status", "source_reference"
]
Constraints_Display = Constraints[[c for c in constraint_cols if c in Constraints.columns]].copy()
display(Constraints_Display)


,constraint_id,constraint_name,threshold_type,threshold_value,denominator,applies_to,implementation_status,source_reference
0,COVERAGE_GLOBAL,Couverture globale des provisions techniques,min,1.0,technical_provisions,all_assets,ENFORCED,Contraintes matérialisées dans le pipeline not...
1,STATE_MIN_20_PT,Titres émis ou garantis par l'État >= 20% PT,min,0.2,technical_provisions,government_bond,ENFORCED,Contraintes matérialisées dans le pipeline not...
2,LISTED_EQUITY_PER_COMPANY_MAX_10_PT,Action cotée BVMT par société <= 10% PT,max,0.1,technical_provisions,listed_equity_per_company,ENFORCED,Contraintes matérialisées dans le pipeline not...
3,REAL_ESTATE_TOTAL_MAX_20_PT,Immobilier total <= 20% PT,max,0.2,technical_provisions,real_estate,POST_CHECK,Contraintes matérialisées dans le pipeline not...
4,SICAR_TOTAL_MAX_10_PT,SICAR/SICAF total <= 10% PT,max,0.1,technical_provisions,sicar_total,POST_CHECK,Contraintes matérialisées dans le pipeline not...
5,CAPITAL_SOCIAL_LIMITS,Limites en pourcentage du capital social,max,NaN,capital_social,issuer,CHECK_NOT_ENFORCED_MISSING_DATA,Contraintes matérialisées dans le pipeline not...
6,OPCVM_ENTITY_LIMITS,OPCVM par entité,max,NaN,technical_provisions,fund_per_entity,CHECK_NOT_ENFORCED_MISSING_DATA,Contraintes matérialisées dans le pipeline not...
7,SICAR_ENTITY_LIMITS,SICAR/SICAF par société,max,NaN,technical_provisions,sicar_per_company,CHECK_NOT_ENFORCED_MISSING_DATA,Contraintes matérialisées dans le pipeline not...
8,OTHER_SECURITIES_LIMITS,Autres valeurs mobilières,max,NaN,technical_provisions,other_securities,CHECK_NOT_ENFORCED_MISSING_DATA,Contraintes matérialisées dans le pipeline not...


## 7. Modèle académique de référence : Markowitz / Mean-Variance

Markowitz est utilisé comme benchmark académique historique. La recommandation finale ne repose pas exclusivement sur la variance, car cette mesure ne capture pas suffisamment le risque extrême important pour une compagnie d'assurances.

Les modèles internes calculés dans cette section incluent Minimum Variance, Mean-Variance avec plusieurs aversions au risque, Maximum Return comme scénario agressif, Mean-CVaR et Risk Parity. Max Sharpe est volontairement exclu.


In [7]:
portfolios, Solver_Diagnostics = solve_all_models(mu_APT, Sigma_APT, Returns_Historical, universe, context, CONFIG)
display(Solver_Diagnostics)


,portfolio_name,success,solver_status,turnover_limit_used,objective_value
0,Minimum_Variance,True,optimal,0.3,0.000078
1,Mean_Variance_lambda_2,True,optimal,0.3,-0.113472
2,Mean_Variance_lambda_5,True,optimal,0.3,-0.112140
3,Mean_Variance_lambda_10,True,optimal,0.3,-0.110053
4,Mean_Variance_lambda_20,True,optimal,0.3,-0.107835
5,Max_Return,True,optimal,0.3,-0.114633
6,MeanVariance_TurnoverPenalty,True,optimal,0.3,-0.098214
7,Mean_CVaR_95,True,optimal,0.3,-0.010389
8,Risk_Parity,True,Optimization terminated successfully,0.3,0.000010


## 8. Frontière efficiente et simulations Monte Carlo

Les simulations Monte Carlo et la frontière efficiente servent à situer les portefeuilles optimisés dans l'espace rendement-risque sous contraintes. Les sélections Monte Carlo excluent Max Sharpe et retiennent uniquement rendement maximal, volatilité minimale, CVaR minimale et scoring multicritère.


In [8]:
Monte_Carlo_All, Monte_Carlo, monte_carlo_weights = generate_monte_carlo(
    mu_APT, Sigma_APT, Returns_Historical, universe, context, Regulatory_Constraints_Map, float(data["rf_annual"]), CONFIG
)
selection_specs = {
    "Monte_Carlo_Max_Return": lambda df: df["expected_return"].idxmax(),
    "Monte_Carlo_Min_Volatility": lambda df: df["volatility"].idxmin(),
    "Monte_Carlo_Min_CVaR": lambda df: df["cvar_95"].idxmin(),
}
selected_rows = []
for label, selector in selection_specs.items():
    idx = selector(Monte_Carlo)
    row = Monte_Carlo.loc[idx].copy()
    pid = str(int(row["portfolio_id"]))
    portfolios[label] = monte_carlo_weights[pid]
    selected_rows.append({
        "Criterion": label,
        "Portfolio_ID": int(row["portfolio_id"]),
        "Expected_Return": float(row["expected_return"]),
        "Volatility": float(row["volatility"]),
        "Sharpe": float(row["sharpe"]),
        "CVaR_95": float(row["cvar_95"]),
    })
Monte_Carlo_Selected = pd.DataFrame(selected_rows)
Monte_Carlo_Selected["Is_Same_As_Other_Selected"] = Monte_Carlo_Selected["Portfolio_ID"].duplicated(keep=False)
if Monte_Carlo_Selected["Is_Same_As_Other_Selected"].any():
    interpretation = "Dans cet univers, certains critères retiennent le même portefeuille ; la volatilité domine alors les métriques de risque."
else:
    interpretation = "Les critères Monte Carlo sélectionnent des portefeuilles distincts."
Monte_Carlo_Selected["Interpretation"] = interpretation
display(Monte_Carlo_Selected)


,Criterion,Portfolio_ID,Expected_Return,Volatility,Sharpe,CVaR_95,Is_Same_As_Other_Selected,Interpretation
0,Monte_Carlo_Max_Return,88395,0.109087,0.019524,1.673798,0.001286,False,Les critères Monte Carlo sélectionnent des por...
1,Monte_Carlo_Min_Volatility,83744,0.107096,0.012956,2.368741,0.000904,False,Les critères Monte Carlo sélectionnent des por...
2,Monte_Carlo_Min_CVaR,73726,0.106954,0.013548,2.254683,0.000829,False,Les critères Monte Carlo sélectionnent des por...


### Frontière efficiente sous contraintes

La frontière dense améliore la lecture du compromis rendement-risque. Les points non convergents sont exclus et les portefeuilles clés sont comparés à la zone efficiente réalisable.


In [9]:
frontier_targets = [float(mu_APT.to_numpy(float) @ np.asarray(w, dtype=float)) for w in portfolios.values()]
Efficient_Frontier = solve_efficient_frontier(
    mu_APT, Sigma_APT, universe, context, float(data["rf_annual"]), CONFIG, target_returns=frontier_targets
)
Efficient_Frontier_Control = pd.DataFrame([{
    "Nombre de points valides": int(len(Efficient_Frontier)),
    "Rendement minimum": float(Efficient_Frontier["achieved_return"].min()),
    "Rendement maximum": float(Efficient_Frontier["achieved_return"].max()),
    "Volatilité minimum": float(Efficient_Frontier["volatility"].min()),
    "Volatilité maximum": float(Efficient_Frontier["volatility"].max()),
}])
display(Efficient_Frontier_Control)
display(Efficient_Frontier[["frontier_point_id", "achieved_return", "volatility", "sharpe"]].head())


,Nombre de points valides,Rendement minimum,Rendement maximum,Volatilité minimum,Volatilité maximum
0,509,0.106696,0.114633,0.00882,0.035049


,frontier_point_id,achieved_return,volatility,sharpe
0,1,0.106696,0.00882,3.433997
1,2,0.106696,0.00882,3.434001
2,3,0.106696,0.00882,3.434013
3,4,0.106697,0.00882,3.434034
4,5,0.106697,0.00882,3.434064


## 9. Comparaison globale des portefeuilles optimisés

Les métriques comparent rendement attendu APT, volatilité, VaR/CVaR, turnover, concentration et statut réglementaire. Maximum Return est interprété comme un scénario agressif, pas comme une recommandation finale.


In [10]:
summary_rows = []
regulatory_tables = []
risk_tables = []
for name, weights in portfolios.items():
    status = "BENCHMARK" if name == "Current_Portfolio" else Solver_Diagnostics.set_index("portfolio_name")["solver_status"].to_dict().get(name, "MONTE_CARLO_SELECTED")
    row, compliance = portfolio_metrics(name, weights, mu_APT, Sigma_APT, Returns_Historical, float(data["rf_annual"]), current_weights, universe, context, Regulatory_Constraints_Map, status)
    row["weights_json"] = json.dumps({asset: float(w) for asset, w in zip(mu_APT.index, weights)}, ensure_ascii=False)
    summary_rows.append(row)
    regulatory_tables.append(compliance)
    try:
        risk_tables.append(risk_contributions(name, weights, Sigma_APT, universe))
    except Exception:
        pass
Optimization_Comparison = pd.DataFrame(summary_rows)
Regulatory_Checks = pd.concat(regulatory_tables, ignore_index=True)
Risk_Contributions = pd.concat(risk_tables, ignore_index=True)
MultiObjective_Scores = build_multiobjective_scores(Optimization_Comparison)
Final_Recommendation = final_recommendation(MultiObjective_Scores)
recommended_name = str(Final_Recommendation.iloc[0]["portfolio_name"])
portfolios["Portefeuille recommande"] = portfolios[recommended_name]
Recommended_Portfolio_Weights = optimized_weights_table({"Portefeuille recommande": portfolios[recommended_name]}, universe, context)
Weights_By_Model = optimized_weights_table(portfolios, universe, context)
Stress_Tests = stress_tests_by_portfolio({k: portfolios[k] for k in ["Current_Portfolio", "Minimum_Variance", "Max_Return", "Mean_CVaR_95", recommended_name] if k in portfolios}, universe, context)

Comparison_Display = Optimization_Comparison[["portfolio_name", "expected_return_APT", "volatility_APT", "sharpe_ratio", "var_95_historical", "cvar_95_historical", "turnover", "regulatory_status"]].sort_values("expected_return_APT", ascending=False)
display(Comparison_Display)
display(Final_Recommendation[["portfolio_name", "final_score", "recommendation_status"]])


,portfolio_name,expected_return_APT,volatility_APT,sharpe_ratio,var_95_historical,cvar_95_historical,turnover,regulatory_status
6,Max_Return,0.114633,0.035049,1.090615,0.001706,0.002424,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
2,Mean_Variance_lambda_2,0.114429,0.030948,1.228574,0.001197,0.002049,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
3,Mean_Variance_lambda_5,0.114289,0.029322,1.291931,0.000906,0.001822,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
4,Mean_Variance_lambda_10,0.113850,0.027560,1.358582,0.000851,0.001658,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
7,MeanVariance_TurnoverPenalty,0.111791,0.027566,1.283580,0.000840,0.001710,0.233535,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
5,Mean_Variance_lambda_20,0.110434,0.016121,2.110715,0.000355,0.000881,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
10,Monte_Carlo_Max_Return,0.109087,0.019524,1.673798,0.000629,0.001286,0.278812,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
8,Mean_CVaR_95,0.109081,0.013718,2.381778,0.000214,0.000503,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
9,Risk_Parity,0.107431,0.010959,2.830905,0.000400,0.000881,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
11,Monte_Carlo_Min_Volatility,0.107096,0.012956,2.368741,0.000457,0.000904,0.287189,COMPLIANT_WITH_UNTESTED_CONSTRAINTS


,portfolio_name,final_score,recommendation_status
8,Mean_CVaR_95,0.582346,SELECTED_BY_MULTICRITERIA_SCORING_NOT_PARETO_S...


## 10. Sensibilité aux scénarios APT

Les scénarios APT prudent, central, optimiste et Historical_Raw permettent de tester la sensibilité des rendements attendus. Historical_Raw reste uniquement descriptif et n'est pas utilisé comme scénario recommandé.


In [11]:
mu_scenarios, APT_Scenarios_Input = load_apt_mu_scenarios(PROJECT_DIR, mu_APT.index.astype(str).tolist())
APT_Scenario_Returns = APT_Scenarios_Input.merge(universe[["asset_id", "asset_name", "asset_class"]], on="asset_id", how="left")
APT_Scenario_Returns = APT_Scenario_Returns.rename(columns={
    "asset_name": "Asset",
    "asset_class": "Class",
    "mu_apt_central": "APT_Return_Central",
    "mu_apt_prudent": "APT_Return_Prudent",
    "mu_apt_optimistic": "APT_Return_Optimistic",
    "mu_historical_raw": "Historical_Raw",
})
APT_Scenario_Returns["Delta_Prudent_vs_Central"] = APT_Scenario_Returns["APT_Return_Prudent"] - APT_Scenario_Returns["APT_Return_Central"]
APT_Scenario_Returns["Delta_Optimistic_vs_Central"] = APT_Scenario_Returns["APT_Return_Optimistic"] - APT_Scenario_Returns["APT_Return_Central"]
if "Historical_Raw" in APT_Scenario_Returns.columns:
    APT_Scenario_Returns["Delta_Historical_vs_Central"] = APT_Scenario_Returns["Historical_Raw"] - APT_Scenario_Returns["APT_Return_Central"]
    APT_Scenario_Returns["Historical_Raw_Comment"] = "Descriptif uniquement ; non utilisé comme scénario principal."
else:
    APT_Scenario_Returns["Historical_Raw"] = np.nan
    APT_Scenario_Returns["Delta_Historical_vs_Central"] = np.nan
    APT_Scenario_Returns["Historical_Raw_Comment"] = "Non disponible dans les exports du notebook 01."

Scenario_Config = replace(CONFIG, slsqp_random_starts=0, frontier_points=400, monte_carlo_required=15_000, monte_carlo_max_attempts=300_000)
APT_Scenario_Analysis = run_apt_scenario_analysis(PROJECT_DIR, Scenario_Config)
Scenario_Comparison = APT_Scenario_Analysis["Scenario_Comparison"]
Scenario_Recommended_Models = APT_Scenario_Analysis["Scenario_Recommended_Models"]
Scenario_Weight_Stability = APT_Scenario_Analysis["Scenario_Weight_Stability"]
Scenario_Recommended_Weights = APT_Scenario_Analysis["Scenario_Recommended_Weights"]
Scenario_Recommended_Performance = APT_Scenario_Analysis["Scenario_Recommended_Performance"]
Scenario_Asset_Sensitivity = APT_Scenario_Analysis["Scenario_Asset_Sensitivity"]
Scenario_Class_Weights = APT_Scenario_Analysis["Scenario_Class_Weights"]
Scenario_Final_Conclusion = APT_Scenario_Analysis["Scenario_Final_Conclusion"]
Factor_Treatment = APT_OPTIMISTIC_FACTOR_TREATMENT.copy()

scenario_display_cols = [
    "Asset", "Class", "APT_Return_Prudent", "APT_Return_Central", "APT_Return_Optimistic",
    "Historical_Raw", "Delta_Prudent_vs_Central", "Delta_Optimistic_vs_Central", "Delta_Historical_vs_Central", "Historical_Raw_Comment"
]
display(APT_Scenario_Returns[[c for c in scenario_display_cols if c in APT_Scenario_Returns.columns]].head(12))
display(Factor_Treatment)
display(Scenario_Recommended_Models)

Scenario_Recommended_Performance_Display = Scenario_Recommended_Performance[[
    "Scenario", "Model", "Expected_Return", "Volatility", "Sharpe", "CVaR_95", "Turnover", "Top_5_Concentration", "Regulatory_Status"
]].copy()
Scenario_Allocation_Distances = Scenario_Weight_Stability.rename(columns={
    "distance_L1_to_central": "Distance_L1_vs_Central",
    "turnover_between_scenarios": "Turnover_vs_Central",
    "weight_stability_score": "Score_Stabilite"
})
Scenario_Recommended_Weights_Display = Scenario_Recommended_Weights.loc[
    Scenario_Recommended_Weights["optimized_weight"].gt(0.005),
    ["Scenario", "asset_name", "asset_class", "optimized_weight"]
].sort_values(["Scenario", "optimized_weight"], ascending=[True, False]).reset_index(drop=True)
Scenario_Recommended_Weights_Pivot = Scenario_Recommended_Weights.pivot_table(
    index=["asset_id", "asset_name", "asset_class"], columns="Scenario", values="optimized_weight", fill_value=0.0
).reset_index()

display(Markdown("**Poids recommandés par scénario**"))
display(Scenario_Recommended_Weights_Display)
display(Markdown("**Distance L1 et turnover entre scénarios**"))
display(Scenario_Allocation_Distances)
display(Markdown("**Rendement / risque du modèle recommandé dans chaque scénario**"))
display(Scenario_Recommended_Performance_Display)


,Asset,Class,APT_Return_Prudent,APT_Return_Central,APT_Return_Optimistic,Historical_Raw,Delta_Prudent_vs_Central,Delta_Optimistic_vs_Central,Delta_Historical_vs_Central,Historical_Raw_Comment
0,"BTA 7,4% 02/2030",Titres de l'État,0.078372,0.100974,0.103586,0.111235,-0.022602,0.002612,0.010261,Descriptif uniquement ; non utilisé comme scén...
1,"BTA 7,5% 12/2028",Titres de l'État,0.079532,0.100101,0.102966,0.108426,-0.020569,0.002865,0.008325,Descriptif uniquement ; non utilisé comme scén...
2,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,0.092617,0.099397,0.102322,0.098383,-0.006780,0.002925,-0.001015,Descriptif uniquement ; non utilisé comme scén...
3,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,0.082846,0.100644,0.103305,0.107073,-0.017798,0.002662,0.006429,Descriptif uniquement ; non utilisé comme scén...
4,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,0.083277,0.101508,0.103981,0.110124,-0.018231,0.002473,0.008616,Descriptif uniquement ; non utilisé comme scén...
5,E.O HL 2020-03,Obligations corporate,0.103641,0.109896,0.117791,0.097950,-0.006255,0.007895,-0.011945,Descriptif uniquement ; non utilisé comme scén...
6,EO BNA SUB 2021-1,Obligations corporate,0.103205,0.108020,0.115892,0.096683,-0.004815,0.007871,-0.011337,Descriptif uniquement ; non utilisé comme scén...
7,EO ADVANS 2022-1,Obligations corporate,0.102621,0.112136,0.119955,0.107498,-0.009515,0.007819,-0.004638,Descriptif uniquement ; non utilisé comme scén...
8,EO ATL 2022-1,Obligations corporate,0.098601,0.111068,0.119045,0.105398,-0.012467,0.007977,-0.005670,Descriptif uniquement ; non utilisé comme scén...
9,EO HL 2023-1,Obligations corporate,0.096414,0.111433,0.119351,0.108759,-0.015019,0.007918,-0.002673,Descriptif uniquement ; non utilisé comme scén...


,Factor,Central_Treatment,Optimistic_Treatment,Kept_Or_Neutralized,Statistical_Reason,Economic_Justification,Impact_Expected
0,mkt_excess_tunindex20,Conservé avec prime de marché annuelle de réfé...,Conserve avec prime de marche moderement relevee,KEPT,Facteur de marché disponible sur l'échantillon...,Un environnement favorable peut soutenir la pr...,Hausse modérée des rendements attendus des act...
1,delta_zc_5y,"Conservé dans la régression, vue directionnell...",Neutralise comme choc directionnel defavorable,NEUTRALIZED_DIRECTIONAL_SHOCK,Facteur de taux fragile sur un échantillon cou...,Un scenario favorable n'ajoute pas de choc de ...,Réduction de la pénalisation des titres obliga...
2,delta_credit_spread_5y,"Conservé dans la régression, vue directionnell...",Neutralise comme choc d'elargissement de spread,NEUTRALIZED_DIRECTIONAL_SHOCK,Spread corporate observé sur un marché peu liq...,Un regime favorable suppose une absence d'elar...,Hausse relative des rendements attendus corpor...
3,delta_inflation_yoy,"Conservé dans la régression, vue directionnell...",Neutralise comme penalisation macro additionnelle,NEUTRALIZED_IF_PENALIZING,Pouvoir explicatif fragile sur une seule année...,La neutralisation evite de sur-penaliser les a...,Diminution d'une pénalité macro additionnelle ...


,Scenario,Recommended_Model,Comment,Recommended_Model_Global
0,APT_Prudent,Mean_CVaR_95,Sélection par scoring multicritère propre au s...,Mean_CVaR_95
1,APT_Central,Mean_CVaR_95,Sélection par scoring multicritère propre au s...,Mean_CVaR_95
2,APT_Optimistic,Mean_CVaR_95,Sélection par scoring multicritère propre au s...,Mean_CVaR_95
3,Historical_Raw,Current_Portfolio,Sélection par scoring multicritère propre au s...,Mean_CVaR_95


**Poids recommandés par scénario**

,Scenario,asset_name,asset_class,optimized_weight
0,APT_Central,EO HL 2023-1,Obligations corporate,0.300000
1,APT_Central,"BTA 7,4% 02/2030",Titres de l'État,0.098109
2,APT_Central,EO AB SUB 2023-2,Obligations corporate,0.093284
3,APT_Central,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,0.087417
4,APT_Central,EO ATL 2025-2,Obligations corporate,0.070822
...,...,...,...,...
68,Historical_Raw,E.O HL 2020-03,Obligations corporate,0.009971
69,Historical_Raw,EO BH LEASING 2023-1,Obligations corporate,0.009783
70,Historical_Raw,EO ATL 2022-1,Obligations corporate,0.008435
71,Historical_Raw,BIAT,Actions cotées,0.006710


**Distance L1 et turnover entre scénarios**

,Scenario,Distance_L1_vs_Central,Turnover_vs_Central,Score_Stabilite
0,APT_Optimistic,0.088624,0.044312,0.955688
1,APT_Prudent,0.308632,0.154316,0.845684
2,Historical_Raw,0.300000,0.150000,0.850000


**Rendement / risque du modèle recommandé dans chaque scénario**

,Scenario,Model,Expected_Return,Volatility,Sharpe,CVaR_95,Turnover,Top_5_Concentration,Regulatory_Status
8,APT_Prudent,Mean_CVaR_95,0.093755,0.013374,1.297071,0.000562,0.3,0.697246,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
8,APT_Central,Mean_CVaR_95,0.109081,0.013718,2.381778,0.000503,0.3,0.649632,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
8,APT_Optimistic,Mean_CVaR_95,0.121922,0.018289,2.488661,0.000723,0.3,0.644367,COMPLIANT_WITH_UNTESTED_CONSTRAINTS
0,Historical_Raw,Current_Portfolio,0.160452,0.015454,5.438369,0.001023,0.0,0.720274,COMPLIANT_WITH_UNTESTED_CONSTRAINTS


## 11. Stress tests principaux

Les stress tests mesurent l'impact de chocs de marché, de taux et de spread sur les portefeuilles clés. Ils complètent les métriques de volatilité et de CVaR.


In [12]:
Stress_Tests_Display = Stress_Tests.groupby(["portfolio_name", "scenario_name"], as_index=False)["estimated_portfolio_impact"].sum()
display(Stress_Tests_Display.head(20))


,portfolio_name,scenario_name,estimated_portfolio_impact
0,Current_Portfolio,Choc actions -10%,-0.012178
1,Current_Portfolio,Choc actions -20%,-0.024356
2,Current_Portfolio,Choc spread corporate +100 bps,-0.011917
3,Current_Portfolio,Choc spread corporate +200 bps,-0.023835
4,Current_Portfolio,Choc taux souverain +100 bps,-0.013057
5,Current_Portfolio,Choc taux souverain +200 bps,-0.026115
6,Max_Return,Choc actions -10%,-0.024458
7,Max_Return,Choc actions -20%,-0.048916
8,Max_Return,Choc spread corporate +100 bps,-0.009304
9,Max_Return,Choc spread corporate +200 bps,-0.018608


## 12. Benchmark externe : HRP / HERC / graph-based via skfolio

La librairie skfolio est utilisée comme outil de benchmark externe. Elle ne remplace pas l'implémentation interne, car le projet nécessite un contrôle spécifique sur les rendements APT, la covariance, la valorisation obligataire, les contraintes réglementaires et la poche optimisable.

Les modèles Max Sharpe, Black-Litterman et les approches trop complexes ne sont pas retenus dans le notebook principal.


In [13]:
Skfolio_Benchmark_Weights, Skfolio_Benchmark_Metrics, Skfolio_Status = run_skfolio_benchmarks(
    Returns_Historical,
    mu_APT,
    Sigma_APT,
    mu_APT.index.astype(str).tolist(),
    current_weights,
    CONFIG,
    universe,
    context,
    Regulatory_Constraints_Map,
    float(data["rf_annual"]),
)

custom_comparison = Optimization_Comparison.loc[
    Optimization_Comparison["portfolio_name"].isin(["Current_Portfolio", recommended_name, "Mean_CVaR_95", "Risk_Parity"]),
    ["portfolio_name", "expected_return_APT", "volatility_APT", "cvar_95_historical", "herfindahl_index", "turnover", "regulatory_status"],
].copy()
custom_comparison = custom_comparison.rename(columns={
    "portfolio_name": "Model",
    "expected_return_APT": "Expected_Return_APT",
    "volatility_APT": "Volatility_APT",
    "cvar_95_historical": "CVaR_95",
    "herfindahl_index": "HHI",
    "turnover": "Turnover_vs_Current",
    "regulatory_status": "Regulatory_Status",
})
custom_comparison["Source"] = "custom"
custom_comparison["Max_Weight"] = custom_comparison["Model"].map({k: float(np.asarray(v).max()) for k, v in portfolios.items()})
custom_comparison["Comment"] = np.where(
    custom_comparison["Model"].eq(recommended_name),
    "Portefeuille recommandé par scoring multicritère interne.",
    "Portefeuille issu du moteur interne validé.",
)
recommended_alias = custom_comparison.loc[custom_comparison["Model"].eq(recommended_name)].copy()
if not recommended_alias.empty:
    recommended_alias["Model"] = "Custom_Recommended"
    recommended_alias["Comment"] = "Portefeuille recommandé par scoring multicritère interne."
    custom_comparison = pd.concat([custom_comparison, recommended_alias], ignore_index=True)

comparison_cols = ["Model", "Source", "Expected_Return_APT", "Volatility_APT", "CVaR_95", "Max_Weight", "HHI", "Turnover_vs_Current", "Regulatory_Status", "Comment"]
skfolio_success = Skfolio_Benchmark_Metrics.loc[Skfolio_Benchmark_Metrics["Status"].eq("PASSED")].copy()
skfolio_not_retained = Skfolio_Benchmark_Metrics.loc[~Skfolio_Benchmark_Metrics["Status"].eq("PASSED")].copy()
Custom_vs_Skfolio_Comparison = pd.concat(
    [custom_comparison[comparison_cols], skfolio_success[[c for c in comparison_cols if c in skfolio_success.columns]]],
    ignore_index=True,
    sort=False,
)
status_message = str(Skfolio_Status["Message"].iloc[0])
if skfolio_success.empty:
    status_message = status_message + " Les feuilles de statut sont exportées et le moteur interne reste la référence."

skfolio_top_weights = pd.DataFrame(columns=["Model", "Asset", "Weight", "asset_name", "asset_class"])
if not Skfolio_Benchmark_Weights.empty:
    skfolio_top_weights = Skfolio_Benchmark_Weights.merge(
        universe[["asset_id", "asset_name", "asset_class"]],
        left_on="Asset",
        right_on="asset_id",
        how="left",
    )
    skfolio_top_weights["asset_name"] = skfolio_top_weights["asset_name"].fillna(skfolio_top_weights["Asset"])
    skfolio_top_weights = skfolio_top_weights.sort_values(["Model", "Weight"], ascending=[True, False]).groupby("Model", as_index=False).head(10)

display(Markdown(f"**Statut skfolio :** {status_message}"))
display(Custom_vs_Skfolio_Comparison)
if not skfolio_not_retained.empty:
    display(Markdown("**Benchmarks skfolio non retenus**"))
    display(skfolio_not_retained[[c for c in ["Model", "Status", "Regulatory_Status", "Reason", "Comment"] if c in skfolio_not_retained.columns]])


**Statut skfolio :** skfolio disponible ; benchmark externe exécuté.

,Model,Source,Expected_Return_APT,Volatility_APT,CVaR_95,Max_Weight,HHI,Turnover_vs_Current,Regulatory_Status,Comment
0,Current_Portfolio,custom,0.106826,0.015454,0.001023,0.416768,0.206067,0.000000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,Portefeuille issu du moteur interne validé.
1,Mean_CVaR_95,custom,0.109081,0.013718,0.000503,0.300000,0.133994,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,Portefeuille recommandé par scoring multicritè...
2,Risk_Parity,custom,0.107431,0.010959,0.000881,0.300000,0.139245,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,Portefeuille issu du moteur interne validé.
3,Custom_Recommended,custom,0.109081,0.013718,0.000503,0.300000,0.133994,0.300000,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,Portefeuille recommandé par scoring multicritè...
4,Skfolio_InverseVolatility,skfolio,0.107373,0.003167,0.000000,0.119569,0.082645,0.972767,NON_TESTABLE_DATA_MISSING,Benchmark externe skfolio ; métriques recalcul...
5,Skfolio_HRP_Variance,skfolio,0.107867,0.002561,0.000000,0.244027,0.123098,1.271326,NON_TESTABLE_DATA_MISSING,Benchmark externe skfolio ; métriques recalcul...
6,Skfolio_Min_SemiVariance,skfolio,0.106993,0.001413,0.000000,0.299277,0.263097,1.695526,NON_TESTABLE_DATA_MISSING,Benchmark externe skfolio ; métriques recalcul...
7,Skfolio_DistributionallyRobustCVaR,skfolio,0.113494,0.036141,0.002617,0.047619,0.047619,0.964357,NON_TESTABLE_DATA_MISSING,Benchmark externe skfolio ; métriques recalcul...


**Benchmarks skfolio non retenus**

,Model,Status,Regulatory_Status,Reason,Comment
1,Skfolio_MaximumDiversification,FAILED,FAILED,NaN,Benchmark externe skfolio ; métriques recalcul...
2,Skfolio_Min_CVaR,FAILED,FAILED,NaN,Benchmark externe skfolio ; métriques recalcul...
3,Skfolio_CVaR_RiskBudgeting,FAILED,NOT_AVAILABLE,"Solver 'SCS' failed. Try another solver, or so...",Échec isolé du benchmark ; le moteur interne r...
4,Skfolio_HRP_CVaR,FAILED,NOT_AVAILABLE,Poids skfolio invalides ou dimension incohéren...,Échec isolé du benchmark ; le moteur interne r...


## 13. Robustesse, turnover et validation multicritère

Mean-CVaR est le modèle principal institutionnel car il contrôle le risque extrême. La variante Mean-CVaR avec contrainte de turnover réduit l'écart par rapport au portefeuille actuel. Robust CVaR / Wasserstein, bootstrap, contribution factorielle et Pareto servent à valider la décision sans remplacer la recommandation interne.


In [14]:
Turnover_Portfolios, Turnover_Constrained_CVaR, Turnover_Constrained_Weights = solve_mean_cvar_turnover_constrained(
    mu_APT, Sigma_APT, Returns_Historical, universe, context, Regulatory_Constraints_Map,
    float(data["rf_annual"]), current_weights, CONFIG, turnover_limits=(0.20, 0.30, 0.40)
)
Robust_CVaR_Wasserstein, Robust_CVaR_Wasserstein_Weights = build_robust_cvar_wasserstein(Skfolio_Benchmark_Metrics, Skfolio_Benchmark_Weights)

def _weights_from_long(weights_df, model_name):
    if weights_df is None or weights_df.empty or "Model" not in weights_df.columns:
        return None
    sub = weights_df.loc[weights_df["Model"].eq(model_name)].copy()
    if sub.empty:
        return None
    asset_col = "Asset" if "Asset" in sub.columns else "asset_id"
    weight_col = "Weight" if "Weight" in sub.columns else "optimized_weight"
    values = sub.set_index(asset_col)[weight_col].astype(float).reindex(mu_APT.index).fillna(0.0).to_numpy(float)
    total = values.sum()
    return values / total if total > 1e-12 else None

turnover_display_cols = ["portfolio_name", "expected_return_APT", "volatility_APT", "var_95_historical", "cvar_95_historical", "turnover", "turnover_limit", "regulatory_status", "Comment"]
Turnover_Constrained_Display = Turnover_Constrained_CVaR[[c for c in turnover_display_cols if c in Turnover_Constrained_CVaR.columns]].copy()
Robust_CVaR_Display = Robust_CVaR_Wasserstein[[c for c in ["Model", "Status", "Expected_Return_APT", "Volatility_APT", "VaR_95", "CVaR_95", "Max_Weight", "Turnover_vs_Current", "Regulatory_Status", "Interpretation"] if c in Robust_CVaR_Wasserstein.columns]].copy()

custom_candidates = Optimization_Comparison.rename(columns={
    "portfolio_name": "Model", "expected_return_APT": "Expected_Return_APT", "volatility_APT": "Volatility_APT",
    "var_95_historical": "VaR_95", "cvar_95_historical": "CVaR_95", "herfindahl_index": "HHI",
    "turnover": "Turnover_vs_Current", "regulatory_status": "Regulatory_Status",
}).copy()
custom_candidates["Source"] = "custom"
custom_candidates["Max_Weight"] = custom_candidates["Model"].map({k: float(np.asarray(v).max()) for k, v in portfolios.items()})

turnover_candidates = Turnover_Constrained_CVaR.rename(columns={
    "portfolio_name": "Model", "expected_return_APT": "Expected_Return_APT", "volatility_APT": "Volatility_APT",
    "var_95_historical": "VaR_95", "cvar_95_historical": "CVaR_95", "herfindahl_index": "HHI",
    "turnover": "Turnover_vs_Current", "regulatory_status": "Regulatory_Status",
}).copy()
turnover_candidates["Source"] = "internal_turnover"
turnover_candidates["Max_Weight"] = turnover_candidates["Model"].map({k: float(np.asarray(v).max()) for k, v in Turnover_Portfolios.items()})

skfolio_candidates = skfolio_success.copy()
skfolio_candidates["Source"] = "skfolio"
robust_candidates = Robust_CVaR_Wasserstein.copy()
robust_candidates["Source"] = "skfolio_robustness"

candidate_cols = ["Model", "Source", "Expected_Return_APT", "Volatility_APT", "VaR_95", "CVaR_95", "Max_Weight", "HHI", "Turnover_vs_Current", "Regulatory_Status", "Comment"]
Enhanced_Candidate_Set = pd.concat(
    [frame[[c for c in candidate_cols if c in frame.columns]] for frame in [custom_candidates, turnover_candidates, skfolio_candidates, robust_candidates] if not frame.empty],
    ignore_index=True, sort=False,
).drop_duplicates(subset=["Model", "Source"], keep="first")

RoRAC_Proxy = build_rorac_proxy(Enhanced_Candidate_Set)
Pareto_Filter = build_pareto_filter(Enhanced_Candidate_Set)

factor_portfolios = {k: portfolios[k] for k in ["Current_Portfolio", "Minimum_Variance", "Mean_Variance_lambda_10", "Mean_CVaR_95", "Risk_Parity", "Max_Return", recommended_name] if k in portfolios}
if "Mean_CVaR_Turnover_Constrained" in Turnover_Portfolios:
    factor_portfolios["Mean_CVaR_Turnover_Constrained"] = Turnover_Portfolios["Mean_CVaR_Turnover_Constrained"]
robust_weights_array = _weights_from_long(Robust_CVaR_Wasserstein_Weights, "Robust_CVaR_Wasserstein")
if robust_weights_array is not None:
    factor_portfolios["Robust_CVaR_Wasserstein"] = robust_weights_array
Factor_Stress_Tests = stress_tests_by_portfolio(factor_portfolios, universe, context)
factor_metrics_base = pd.concat([Optimization_Comparison, Turnover_Constrained_CVaR], ignore_index=True, sort=False)
if robust_weights_array is not None:
    robust_metric_row, _ = portfolio_metrics("Robust_CVaR_Wasserstein", robust_weights_array, mu_APT, Sigma_APT, Returns_Historical, float(data["rf_annual"]), current_weights, universe, context, Regulatory_Constraints_Map, "BENCHMARK_EXTERNAL")
    factor_metrics_base = pd.concat([factor_metrics_base, pd.DataFrame([robust_metric_row])], ignore_index=True, sort=False)
key_factor_models = list(dict.fromkeys(["Current_Portfolio", "Minimum_Variance", "Mean_CVaR_95", "Mean_CVaR_Turnover_Constrained", "Risk_Parity", "Robust_CVaR_Wasserstein", recommended_name]))
Factor_Risk_Contribution = build_factor_risk_sensitivity(factor_metrics_base, Factor_Stress_Tests, key_factor_models)

Bootstrap_Asset_Stability, Bootstrap_Stability_Check = bootstrap_stability_check(Returns_Historical, mu_APT, portfolios[recommended_name], universe, n_bootstrap=300, random_seed=CONFIG.random_seed)

Final_Scoring_With_Robustness = Enhanced_Candidate_Set.merge(
    RoRAC_Proxy[["Model", "Return_to_CVaR", "Return_to_RiskPenalty"]], on="Model", how="left"
).merge(
    Pareto_Filter[["Model", "Is_Pareto_Efficient", "Dominated_By", "Pareto_Comment"]], on="Model", how="left"
)
Final_Scoring_With_Robustness["Recommendation_Use"] = np.where(
    Final_Scoring_With_Robustness["Source"].astype(str).str.contains("skfolio", case=False, na=False),
    "Benchmark externe uniquement",
    np.where(Final_Scoring_With_Robustness["Model"].eq(recommended_name), "Recommandation interne", "Candidat interne ou validation"),
)

role_map = {
    "Current_Portfolio": "Current Portfolio", "Minimum_Variance": "Academic Benchmark",
    "Mean_Variance_lambda_2": "Academic Benchmark", "Mean_Variance_lambda_5": "Academic Benchmark",
    "Mean_Variance_lambda_10": "Academic Benchmark", "Mean_Variance_lambda_20": "Academic Benchmark",
    "Max_Return": "Scenario agressif / borne supérieure", "Mean_CVaR_95": "Main Institutional Model",
    "Mean_CVaR_Turnover_Constrained": "Operational Variant", "Risk_Parity": "Risk Diversification",
    "Robust_CVaR_Wasserstein": "Robustness Test",
}
Final_Comparison_Clean = Final_Scoring_With_Robustness.copy()
role_default = pd.Series(np.where(Final_Comparison_Clean["Source"].astype(str).str.contains("skfolio", case=False, na=False), "External Benchmark", "Monte Carlo Candidate"), index=Final_Comparison_Clean.index)
Final_Comparison_Clean["Role"] = Final_Comparison_Clean["Model"].map(role_map).fillna(role_default)
Final_Comparison_Clean["RoRAC_Proxy"] = Final_Comparison_Clean["Return_to_RiskPenalty"]
Final_Comparison_Clean["Pareto_Status"] = np.where(Final_Comparison_Clean["Is_Pareto_Efficient"].fillna(False), "PARETO_EFFICIENT", "DOMINATED_OR_NOT_TESTED")
bootstrap_status = str(Bootstrap_Stability_Check["Bootstrap_Stability_Status"].iloc[0]) if not Bootstrap_Stability_Check.empty else "NOT_AVAILABLE"
Final_Comparison_Clean["Bootstrap_Stability"] = np.where(Final_Comparison_Clean["Model"].eq(recommended_name), bootstrap_status, "")
Final_Comparison_Clean["Recommended"] = np.where(Final_Comparison_Clean["Model"].eq(recommended_name), "YES", "NO")
Final_Comparison_Clean["Comment"] = Final_Comparison_Clean["Comment"].fillna(Final_Comparison_Clean["Recommendation_Use"])
final_cols = ["Model", "Role", "Source", "Expected_Return_APT", "Volatility_APT", "VaR_95", "CVaR_95", "Max_Weight", "HHI", "Turnover_vs_Current", "RoRAC_Proxy", "Pareto_Status", "Bootstrap_Stability", "Regulatory_Status", "Recommended", "Comment"]
Final_Comparison_Clean = Final_Comparison_Clean[[c for c in final_cols if c in Final_Comparison_Clean.columns]].sort_values(["Recommended", "Role", "Expected_Return_APT"], ascending=[False, True, False]).reset_index(drop=True)

Markowitz_Benchmark = Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].str.contains("Minimum_Variance|Mean_Variance", regex=True)].copy()
Mean_CVaR_Summary = Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].eq("Mean_CVaR_95")].copy()
Risk_Parity_Summary = Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].eq("Risk_Parity")].copy()
Risk_Parity_Contribution = Risk_Contributions.loc[Risk_Contributions["portfolio_name"].eq("Risk_Parity")].copy()
Skfolio_Benchmarks = Skfolio_Benchmark_Metrics.copy()
Skfolio_Benchmarks_Not_Retained = skfolio_not_retained.copy()

Final_Clean_Summary = pd.DataFrame([
    {"Item": "Moteur de recommandation", "Value": "Interne", "Comment": "APT, covariance APT, contraintes testables, Mean-CVaR et scoring multicritère."},
    {"Item": "Markowitz", "Value": "Benchmark académique", "Comment": "Conservé comme référence historique rendement-risque."},
    {"Item": "Mean-CVaR", "Value": "Modèle principal", "Comment": "Risque extrême plus pertinent qu'une variance seule pour une compagnie d'assurances."},
    {"Item": "Mean-CVaR turnover", "Value": "Variante opérationnelle", "Comment": "Réduit la rotation excessive par rapport au portefeuille actuel."},
    {"Item": "Robust CVaR / Wasserstein", "Value": "Test de robustesse", "Comment": "Non recommandé automatiquement."},
    {"Item": "skfolio", "Value": "Benchmark externe", "Comment": "Ne remplace pas le moteur interne."},
    {"Item": "Bootstrap", "Value": "Stabilité", "Comment": "Évalue la stabilité des métriques ; pas un modèle d'allocation."},
])

display(Markdown("**Mean-CVaR avec contrainte de turnover**"))
display(Turnover_Constrained_Display)
display(Markdown("**Robust CVaR / Wasserstein**"))
display(Robust_CVaR_Display)
display(Markdown("**Synthèse multicritère finale**"))
display(Final_Comparison_Clean.loc[Final_Comparison_Clean["Model"].isin([recommended_name, "Mean_CVaR_95", "Mean_CVaR_Turnover_Constrained", "Risk_Parity", "Robust_CVaR_Wasserstein"])])


**Mean-CVaR avec contrainte de turnover**

,portfolio_name,expected_return_APT,volatility_APT,var_95_historical,cvar_95_historical,turnover,turnover_limit,regulatory_status,Comment
0,Mean_CVaR_Turnover_20,0.106826,0.015454,0.000499,0.001023,0.0,0.2,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,Non retenu : solveur non convergent sous la li...
1,Mean_CVaR_Turnover_Constrained,0.108712,0.012421,0.000181,0.000476,0.3,0.3,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,CVaR avec contrainte explicite de turnover ; v...
2,Mean_CVaR_Turnover_40,0.108356,0.005768,0.000000,0.000091,0.4,0.4,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,CVaR avec contrainte explicite de turnover ; v...


**Robust CVaR / Wasserstein**

,Model,Status,Expected_Return_APT,Volatility_APT,VaR_95,CVaR_95,Max_Weight,Turnover_vs_Current,Regulatory_Status,Interpretation
7,Robust_CVaR_Wasserstein,PASSED,0.113494,0.036141,0.00157,0.002617,0.047619,0.964357,NON_TESTABLE_DATA_MISSING,Benchmark robuste de CVaR sous incertitude de ...


**Synthèse multicritère finale**

,Model,Role,Source,Expected_Return_APT,Volatility_APT,VaR_95,CVaR_95,Max_Weight,HHI,Turnover_vs_Current,RoRAC_Proxy,Pareto_Status,Bootstrap_Stability,Regulatory_Status,Recommended,Comment
0,Mean_CVaR_95,Main Institutional Model,custom,0.109081,0.013718,0.000214,0.000503,0.300000,0.133994,0.300000,9.754553,PARETO_EFFICIENT,PERFORMANCE_STABILITY_EVALUATED,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,YES,Recommandation interne
17,Mean_CVaR_Turnover_Constrained,Operational Variant,internal_turnover,0.108712,0.012421,0.000181,0.000476,0.300000,0.133998,0.300000,9.745169,PARETO_EFFICIENT,,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,NO,CVaR avec contrainte explicite de turnover ; v...
18,Risk_Parity,Risk Diversification,custom,0.107431,0.010959,0.000400,0.000881,0.300000,0.139245,0.300000,9.208785,PARETO_EFFICIENT,,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,NO,Candidat interne ou validation
19,Robust_CVaR_Wasserstein,Robustness Test,skfolio_robustness,0.113494,0.036141,0.001570,0.002617,0.047619,0.047619,0.964357,12.547590,PARETO_EFFICIENT,,NON_TESTABLE_DATA_MISSING,NO,Benchmark externe skfolio ; métriques recalcul...


## 14. Graphiques décisionnels

Les figures visibles sont limitées aux graphiques utiles à la décision : composition, rendement-risque, corrélation, frontière efficiente, allocations, risque extrême, turnover, contribution au risque, benchmark externe, Pareto, scénarios APT et stabilité bootstrap.


In [15]:
exported_figures = []

def _clean_layout(fig, height=560):
    fig.update_layout(template="plotly_white", height=height, margin=dict(l=40, r=30, t=80, b=60), legend_title_text="")
    return fig

composition = universe.groupby("asset_class", as_index=False)["current_weight_optimisable"].sum()
fig = px.bar(composition, x="asset_class", y="current_weight_optimisable", color="asset_class", text=composition["current_weight_optimisable"].map(lambda x: f"{x:.1%}"), title="Composition actuelle de la poche optimisable")
fig.update_yaxes(tickformat=".1%", title="Poids")
fig.update_xaxes(title="Classe d'actifs")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "01_composition_poche_optimisable"))

fig = px.scatter(Asset_Risk_Interpretation, x="historical_volatility", y="expected_return_APT", color="asset_class", hover_name="asset_name", title="Rendement attendu APT vs volatilité historique des actifs")
fig.update_xaxes(tickformat=".1%", title="Volatilité historique annualisée")
fig.update_yaxes(tickformat=".1%", title="Rendement attendu APT")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "02_rendement_apt_volatilite_actifs"))

fig = px.imshow(Correlation, color_continuous_scale="RdBu", zmin=-1, zmax=1, title="Matrice de corrélation des actifs optimisables")
exported_figures.append(show_and_export_fig(_clean_layout(fig, height=650), "03_correlation_actifs_optimisables"))

fig = go.Figure()
mc_sample = Monte_Carlo.sample(min(len(Monte_Carlo), 12000), random_state=CONFIG.random_seed) if len(Monte_Carlo) > 12000 else Monte_Carlo
fig.add_trace(go.Scatter(x=mc_sample["volatility"], y=mc_sample["expected_return"], mode="markers", name="Monte Carlo faisable", marker=dict(size=3, color="rgba(70,70,70,0.18)"), customdata=mc_sample["cvar_95"], hovertemplate="Rendement %{y:.2%}<br>Volatilité %{x:.2%}<br>CVaR %{customdata:.2%}<extra></extra>"))
frontier_plot = Efficient_Frontier.sort_values("volatility").drop_duplicates(subset=["volatility", "achieved_return"])
fig.add_trace(go.Scatter(x=frontier_plot["volatility"], y=frontier_plot["achieved_return"], mode="lines", name=f"Frontière efficiente ({len(frontier_plot)} points)", line=dict(width=4, color="#005f73"), hovertemplate="Frontière<br>Rendement %{y:.2%}<br>Volatilité %{x:.2%}<extra></extra>"))
frontier_models = ["Current_Portfolio", "Minimum_Variance", "Mean_Variance_lambda_10", "Mean_CVaR_95", "Mean_CVaR_Turnover_Constrained", "Risk_Parity", "Robust_CVaR_Wasserstein", recommended_name]
plot_models = Final_Comparison_Clean.loc[Final_Comparison_Clean["Model"].isin(frontier_models)].drop_duplicates("Model").copy()
fig.add_trace(go.Scatter(x=plot_models["Volatility_APT"], y=plot_models["Expected_Return_APT"], mode="markers+text", text=plot_models["Model"], textposition="top center", name="Portefeuilles clés", marker=dict(size=13, color="#ca6702", line=dict(width=1, color="white")), customdata=plot_models[["CVaR_95", "HHI", "Turnover_vs_Current", "Regulatory_Status"]].to_numpy(), hovertemplate="%{text}<br>Rendement %{y:.2%}<br>Volatilité %{x:.2%}<br>CVaR %{customdata[0]:.2%}<br>HHI %{customdata[1]:.3f}<br>Turnover %{customdata[2]:.2f}<br>%{customdata[3]}<extra></extra>"))
fig.update_xaxes(tickformat=".1%", title="Volatilité annualisée")
fig.update_yaxes(tickformat=".1%", title="Rendement attendu annualisé")
fig.update_layout(title="Frontière efficiente, simulations Monte Carlo et portefeuilles optimisés")
exported_figures.append(show_and_export_fig(_clean_layout(fig, height=760), "04_frontiere_efficiente_monte_carlo_modeles"))

main_weight_models = ["Current_Portfolio", "Minimum_Variance", "Mean_CVaR_95", "Mean_CVaR_Turnover_Constrained", "Risk_Parity", recommended_name]
weights_internal = Weights_By_Model.loc[Weights_By_Model["portfolio_name"].isin([m for m in main_weight_models if m in Weights_By_Model["portfolio_name"].unique()])].copy()
tw = Turnover_Constrained_Weights.copy()
if {"portfolio_name", "asset_id", "optimized_weight"}.issubset(tw.columns):
    tw = tw.loc[tw["portfolio_name"].eq("Mean_CVaR_Turnover_Constrained")].merge(universe[["asset_id", "asset_class", "asset_name"]], on="asset_id", how="left")
    weights_internal = pd.concat([weights_internal, tw[[c for c in weights_internal.columns if c in tw.columns]]], ignore_index=True, sort=False)
weights_class = weights_internal.groupby(["portfolio_name", "asset_class"], as_index=False)["optimized_weight"].sum()
fig = px.bar(weights_class, x="portfolio_name", y="optimized_weight", color="asset_class", title="Allocations comparées des modèles principaux")
fig.update_yaxes(tickformat=".1%", title="Poids")
fig.update_xaxes(title="Modèle", tickangle=30)
exported_figures.append(show_and_export_fig(_clean_layout(fig), "05_allocations_modeles_principaux"))

risk_compare = Final_Comparison_Clean.loc[Final_Comparison_Clean["Model"].isin(frontier_models), ["Model", "Expected_Return_APT", "Volatility_APT", "CVaR_95"]].copy()
risk_long = risk_compare.melt(id_vars="Model", var_name="Metric", value_name="Value")
fig = px.bar(risk_long, x="Model", y="Value", color="Metric", barmode="group", title="Rendement, volatilité et CVaR des portefeuilles clés")
fig.update_yaxes(tickformat=".1%")
fig.update_xaxes(tickangle=30)
exported_figures.append(show_and_export_fig(_clean_layout(fig), "06_rendement_volatilite_cvar_modeles"))

fig = px.scatter(Turnover_Constrained_CVaR, x="turnover", y="cvar_95_historical", color="turnover_limit", size="expected_return_APT", hover_name="portfolio_name", title="Mean-CVaR : effet de la contrainte de turnover")
fig.update_xaxes(tickformat=".1%", title="Turnover vs portefeuille actuel")
fig.update_yaxes(tickformat=".2%", title="CVaR 95 %")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "07_mean_cvar_vs_turnover"))

rec_risk = Risk_Contributions.loc[Risk_Contributions["portfolio_name"].eq(recommended_name)].sort_values("risk_contribution_pct", ascending=False).head(12)
fig = px.bar(rec_risk, x="asset_name", y="risk_contribution_pct", color="asset_class", title="Contribution au risque par actif du portefeuille recommandé")
fig.update_yaxes(tickformat=".1%", title="Contribution au risque")
fig.update_xaxes(tickangle=35, title="Actif")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "08_contribution_risque_actif_recommande"))

factor_cols = ["Market_Factor_Contribution", "Rate_Factor_Contribution", "Credit_Spread_Contribution", "Inflation_Contribution", "Specific_Risk_Contribution"]
factor_long = Factor_Risk_Contribution.melt(id_vars="Portfolio", value_vars=[c for c in factor_cols if c in Factor_Risk_Contribution.columns], var_name="Facteur", value_name="Contribution")
fig = px.bar(factor_long, x="Portfolio", y="Contribution", color="Facteur", title="Contribution au risque par facteur APT ou choc de facteur")
fig.update_yaxes(tickformat=".2%", title="Contribution estimée")
fig.update_xaxes(tickangle=30)
exported_figures.append(show_and_export_fig(_clean_layout(fig), "09_contribution_risque_facteur_apt"))

bench_plot = Custom_vs_Skfolio_Comparison.dropna(subset=["Expected_Return_APT", "Volatility_APT"]).copy()
fig = px.scatter(bench_plot, x="Volatility_APT", y="Expected_Return_APT", color="Source", size="HHI", hover_name="Model", title="Benchmark externe skfolio vs moteur interne")
fig.update_xaxes(tickformat=".1%", title="Volatilité APT")
fig.update_yaxes(tickformat=".1%", title="Rendement attendu APT")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "10_benchmark_custom_vs_skfolio"))

pareto_plot = Pareto_Filter.dropna(subset=["Expected_Return_APT", "CVaR_95", "HHI"]).copy()
fig = px.scatter(pareto_plot, x="CVaR_95", y="Expected_Return_APT", color="Is_Pareto_Efficient", size="HHI", symbol="Source", hover_name="Model", title="Filtre Pareto rendement - risque extrême - concentration")
fig.update_xaxes(tickformat=".2%", title="CVaR 95 %")
fig.update_yaxes(tickformat=".1%", title="Rendement attendu APT")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "11_filtre_pareto"))

scenario_cols = ["APT_Return_Prudent", "APT_Return_Central", "APT_Return_Optimistic", "Historical_Raw"]
scenario_long = APT_Scenario_Returns.melt(id_vars=["Asset", "Class"], value_vars=[c for c in scenario_cols if c in APT_Scenario_Returns.columns], var_name="Scenario", value_name="Expected_Return")
fig = px.box(scenario_long, x="Scenario", y="Expected_Return", color="Scenario", points="all", hover_name="Asset", title="Scénarios APT et scénario Historical_Raw")
fig.update_yaxes(tickformat=".1%", title="Rendement attendu annualisé")
fig.update_xaxes(title="Scénario")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "12_scenarios_apt_historical_raw"))

boot_top = Bootstrap_Asset_Stability.sort_values("Mean_Weight", ascending=False).head(10).copy()
fig = px.bar(boot_top, x="asset_name", y="Mean_Weight", error_y="Std_Weight", color="asset_class", title="Bootstrap stability : poids du portefeuille recommandé")
fig.update_yaxes(tickformat=".1%", title="Poids moyen")
fig.update_xaxes(tickangle=35, title="Actif")
exported_figures.append(show_and_export_fig(_clean_layout(fig), "13_bootstrap_stability_recommande"))

Figure_Exports = pd.DataFrame({"Figure": [p.stem for p in exported_figures], "Path": [project_relative(p) for p in exported_figures]})
Figure_Exports_Summary = pd.DataFrame([{"Nombre de graphiques visibles": len(Figure_Exports), "Dossier HTML": project_relative(FIGURES_DIR)}])
display(Figure_Exports_Summary)


,Nombre de graphiques visibles,Dossier HTML
0,13,exports\figures\notebook_02


## 15. Filtre Pareto et scoring multicritère

Le scoring final reste fondé sur les candidats internes. Les métriques de robustesse, le filtre Pareto et le proxy rendement-risque documentent la décision, sans laisser un benchmark externe remplacer automatiquement le portefeuille retenu.


In [16]:
Scoring_Final_Display = Final_Comparison_Clean.loc[
    Final_Comparison_Clean["Model"].isin([recommended_name, "Mean_CVaR_95", "Mean_CVaR_Turnover_Constrained", "Risk_Parity", "Max_Return", "Robust_CVaR_Wasserstein"])
].copy()
display(Scoring_Final_Display)


,Model,Role,Source,Expected_Return_APT,Volatility_APT,VaR_95,CVaR_95,Max_Weight,HHI,Turnover_vs_Current,RoRAC_Proxy,Pareto_Status,Bootstrap_Stability,Regulatory_Status,Recommended,Comment
0,Mean_CVaR_95,Main Institutional Model,custom,0.109081,0.013718,0.000214,0.000503,0.300000,0.133994,0.300000,9.754553,PARETO_EFFICIENT,PERFORMANCE_STABILITY_EVALUATED,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,YES,Recommandation interne
17,Mean_CVaR_Turnover_Constrained,Operational Variant,internal_turnover,0.108712,0.012421,0.000181,0.000476,0.300000,0.133998,0.300000,9.745169,PARETO_EFFICIENT,,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,NO,CVaR avec contrainte explicite de turnover ; v...
18,Risk_Parity,Risk Diversification,custom,0.107431,0.010959,0.000400,0.000881,0.300000,0.139245,0.300000,9.208785,PARETO_EFFICIENT,,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,NO,Candidat interne ou validation
19,Robust_CVaR_Wasserstein,Robustness Test,skfolio_robustness,0.113494,0.036141,0.001570,0.002617,0.047619,0.047619,0.964357,12.547590,PARETO_EFFICIENT,,NON_TESTABLE_DATA_MISSING,NO,Benchmark externe skfolio ; métriques recalcul...
20,Max_Return,Scenario agressif / borne supérieure,custom,0.114633,0.035049,0.001706,0.002424,0.300000,0.132520,0.300000,8.767501,PARETO_EFFICIENT,,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,NO,Candidat interne ou validation


## 16. Portefeuille recommandé

Le portefeuille recommandé résulte du scoring multicritère interne. Les benchmarks externes et tests de robustesse servent à confirmer ou nuancer la recommandation, pas à la remplacer automatiquement.


In [17]:
Recommended_Portfolio = Final_Recommendation.copy()
Recommended_Allocation_Display = Recommended_Portfolio_Weights.loc[
    Recommended_Portfolio_Weights["optimized_weight"].gt(0.001),
    ["asset_name", "asset_class", "optimized_weight", "optimized_value", "weight_change"]
].sort_values("optimized_weight", ascending=False)
Regulatory_Readable = Regulatory_Checks.copy()
Regulatory_Readable["Financial_Interpretation"] = np.where(
    Regulatory_Readable["compliance_status"].astype(str).str.contains("MISSING|NOT_ENFORCED", case=False, na=False),
    "Certaines données nécessaires ne sont pas disponibles ; la contrainte est documentée mais non conclue.",
    np.where(Regulatory_Readable["compliance_status"].astype(str).str.contains("BREACH", case=False, na=False), "Violation détectée sur une contrainte testable.", "Aucune violation détectée sur cette contrainte testable."),
)
Recommended_Final_Table = Final_Comparison_Clean.loc[Final_Comparison_Clean["Recommended"].eq("YES")].copy()
display(Recommended_Final_Table)
display(Recommended_Allocation_Display)
regulatory_display_cols = ["portfolio_name", "constraint_id", "constraint_name", "exposure_value", "threshold_value", "exposure_pct_denominator", "limit_pct", "compliance_status", "Financial_Interpretation"]
display(Regulatory_Readable[[c for c in regulatory_display_cols if c in Regulatory_Readable.columns]].head(20))


,Model,Role,Source,Expected_Return_APT,Volatility_APT,VaR_95,CVaR_95,Max_Weight,HHI,Turnover_vs_Current,RoRAC_Proxy,Pareto_Status,Bootstrap_Stability,Regulatory_Status,Recommended,Comment
0,Mean_CVaR_95,Main Institutional Model,custom,0.109081,0.013718,0.000214,0.000503,0.3,0.133994,0.3,9.754553,PARETO_EFFICIENT,PERFORMANCE_STABILITY_EVALUATED,COMPLIANT_WITH_UNTESTED_CONSTRAINTS,YES,Recommandation interne


,asset_name,asset_class,optimized_weight,optimized_value,weight_change
9,EO HL 2023-1,Obligations corporate,0.300000,9.925947e+07,-1.167675e-01
0,"BTA 7,4% 02/2030",Titres de l'État,0.098109,3.246080e+07,1.668267e-09
11,EO AB SUB 2023-2,Obligations corporate,0.093284,3.086435e+07,8.710950e-02
2,EMPRUNT NATIONAL 2021 2ème tranche,Titres de l'État,0.087417,2.892336e+07,1.453991e-09
12,EO ATL 2025-2,Obligations corporate,0.070822,2.343258e+07,3.878746e-02
1,"BTA 7,5% 12/2028",Titres de l'État,0.068296,2.259674e+07,1.527128e-09
4,EMPRUNT NATIONAL 2023 4ème tranche Catégorie C,Titres de l'État,0.049684,1.643861e+07,1.778959e-09
3,EMPRUNT NATIONAL 2022 3ème tranche 10 ANS,Titres de l'État,0.043850,1.450828e+07,1.620295e-09
20,PGH,Actions cotées,0.036801,1.217614e+07,-8.058011e-03
7,EO ADVANS 2022-1,Obligations corporate,0.028664,9.483798e+06,2.207770e-07


,portfolio_name,constraint_id,constraint_name,exposure_value,threshold_value,exposure_pct_denominator,limit_pct,compliance_status,Financial_Interpretation
0,Current_Portfolio,COVERAGE_GLOBAL,Couverture globale des provisions techniques,5.090326e+08,2.611640e+08,1.949091,1.0,COMPLIANT,Aucune violation détectée sur cette contrainte...
1,Current_Portfolio,STATE_MIN_20_PT,Titres émis ou garantis par l'État >= 20% PT,1.149278e+08,5.223281e+07,0.440060,0.2,COMPLIANT,Aucune violation détectée sur cette contrainte...
2,Current_Portfolio,REAL_ESTATE_TOTAL_MAX_20_PT,Immobilier total <= 20% PT,2.586088e+07,5.223281e+07,0.099022,0.2,COMPLIANT,Aucune violation détectée sur cette contrainte...
3,Current_Portfolio,SICAR_TOTAL_MAX_10_PT,SICAR/SICAF total <= 10% PT,5.215649e+06,2.611640e+07,0.019971,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
4,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::AMEN_BANK,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::AMEN_BANK,7.042400e+06,2.611640e+07,0.026965,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
5,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::ATTIJARI_...,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::ATTIJARI_...,5.382839e+06,2.611640e+07,0.020611,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
6,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::BIAT,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::BIAT,2.220000e+06,2.611640e+07,0.008500,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
7,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::BT,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::BT,4.480000e+04,2.611640e+07,0.000172,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
8,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::CITY_CARS,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::CITY_CARS,3.531619e+05,2.611640e+07,0.001352,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...
9,Current_Portfolio,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::DELICE_HO...,LISTED_EQUITY_PER_COMPANY_MAX_10_PT::DELICE_HO...,1.407932e+06,2.611640e+07,0.005391,0.1,COMPLIANT,Aucune violation détectée sur cette contrainte...


## 17. Exports

Les résultats principaux sont exportés en Excel et les figures en HTML. Les détails techniques complets restent dans les exports et dans le rapport qualité séparé.


In [18]:
Final_Summary = pd.DataFrame([
    {"Item": "Modèle recommandé", "Value": recommended_name, "Comment": "Sélection par scoring multicritère interne."},
    {"Item": "Rendement attendu recommandé", "Value": float(Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].eq(recommended_name), "expected_return_APT"].iloc[0]), "Comment": "Rendement attendu APT central."},
    {"Item": "Risque recommandé", "Value": float(Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].eq(recommended_name), "volatility_APT"].iloc[0]), "Comment": "Volatilité APT annualisée."},
    {"Item": "CVaR historique recommandée", "Value": float(Optimization_Comparison.loc[Optimization_Comparison["portfolio_name"].eq(recommended_name), "cvar_95_historical"].iloc[0]), "Comment": "Perte historique extrême, utilisée en contrôle."},
    {"Item": "Contraintes testables", "Value": "Aucune violation détectée" if not Regulatory_Checks["compliance_status"].astype(str).str.contains("BREACH", case=False, na=False).any() else "Violation détectée", "Comment": "Certaines contraintes restent non testables faute de données détaillées."},
    {"Item": "Historical_Raw", "Value": "Comparaison uniquement", "Comment": "Non utilisé comme scénario principal."},
    {"Item": "Max Sharpe", "Value": "Exclu", "Comment": "Ratio instable dans un univers avec volatilités obligataires lissées."},
    {"Item": "Frontière efficiente", "Value": len(Efficient_Frontier), "Comment": "Points valides sous contraintes."},
    {"Item": "Monte Carlo", "Value": len(Monte_Carlo), "Comment": "Portefeuilles faisables retenus."},
    {"Item": "Base notebook 03", "Value": "OK", "Comment": "Résultats utilisables pour l'allocation additionnelle de 10 MD."},
])
Conclusion_Export = pd.DataFrame([
    {"Conclusion": "Markowitz est conservé comme benchmark académique. Mean-CVaR reste le modèle principal institutionnel, complété par une variante turnover, des tests de robustesse, un filtre Pareto, un bootstrap et des benchmarks skfolio externes. La recommandation finale reste issue du moteur interne."}
])

exports = {
    "01_Hypotheses": Hypotheses,
    "02_Current_Portfolio": Current_Portfolio,
    "03_APT_Expected_Returns": APT_Scenario_Returns,
    "04_Constraints": Regulatory_Readable,
    "05_Markowitz_Benchmark": Markowitz_Benchmark,
    "06_Mean_CVaR": Mean_CVaR_Summary,
    "07_Turnover_CVaR": Turnover_Constrained_CVaR,
    "08_Risk_Parity": Risk_Parity_Contribution if not Risk_Parity_Contribution.empty else Risk_Parity_Summary,
    "09_Factor_Risk_Contribution": Factor_Risk_Contribution,
    "10_Robust_CVaR": Robust_CVaR_Wasserstein,
    "11_Bootstrap_Stability": pd.concat([Bootstrap_Stability_Check, Bootstrap_Asset_Stability], ignore_index=True, sort=False),
    "12_Skfolio_Benchmarks": Skfolio_Benchmarks,
    "13_Monte_Carlo": Monte_Carlo,
    "14_Efficient_Frontier": Efficient_Frontier,
    "15_Pareto_Filter": Pareto_Filter,
    "16_Final_Comparison": Final_Comparison_Clean,
    "17_Recommended_Portfolio": Recommended_Portfolio_Weights,
    "18_Conclusion": Conclusion_Export,
    "Hypotheses": Hypotheses,
    "Current_Portfolio": Current_Portfolio,
    "Expected_Returns_APT": APT_Scenario_Returns,
    "Constraints": Regulatory_Readable,
    "Monte_Carlo": Monte_Carlo,
    "Efficient_Frontier": Efficient_Frontier,
    "Recommended_Portfolio": Recommended_Portfolio_Weights,
    "Inputs": Inputs,
    "Asset_Metrics": Asset_Risk_Interpretation,
    "Covariance": Covariance.reset_index().rename(columns={"index": "asset_id"}),
    "Optimized_Portfolios": Weights_By_Model,
    "Monte_Carlo_Selected": Monte_Carlo_Selected,
    "Scenario_Analysis": Scenario_Comparison,
    "Scenario_Recommended_Weights": Scenario_Recommended_Weights,
    "Scenario_Recommended_Perf": Scenario_Recommended_Performance_Display,
    "Scenario_Allocation_Dist": Scenario_Allocation_Distances,
    "Scenario_Recommended": Scenario_Recommended_Models,
    "Scenario_Weight_Stability": Scenario_Weight_Stability,
    "Scenario_Factor_Treatment": Factor_Treatment,
    "Stress_Tests": Stress_Tests,
    "Factor_Stress_Tests": Factor_Stress_Tests,
    "Skfolio_Benchmark_Weights": Skfolio_Benchmark_Weights,
    "Skfolio_Not_Retained": Skfolio_Benchmarks_Not_Retained,
    "Custom_vs_Skfolio": Custom_vs_Skfolio_Comparison,
    "Skfolio_Status": Skfolio_Status,
    "Robust_CVaR_Weights": Robust_CVaR_Wasserstein_Weights,
    "Turnover_CVaR_Weights": Turnover_Constrained_Weights,
    "RoRAC_Proxy": RoRAC_Proxy,
    "Bootstrap_Asset_Stability": Bootstrap_Asset_Stability,
    "Final_Clean_Summary": Final_Clean_Summary,
    "Final_Summary": Final_Summary,
    "Figure_Exports": Figure_Exports,
}
with pd.ExcelWriter(EXCEL_OUTPUT, engine="openpyxl") as writer:
    for sheet, df in exports.items():
        out = df.copy() if isinstance(df, pd.DataFrame) else pd.DataFrame(df)
        if "weights_array" in out.columns:
            out = out.drop(columns=["weights_array"])
        out.to_excel(writer, sheet_name=sheet[:31], index=False)

Scenario_Excel_Export_Path = export_apt_scenario_sheets(APT_Scenario_Analysis, EXCEL_OUTPUT)
Export_Control = pd.DataFrame([
    {"Export": "Excel", "Path": project_relative(EXCEL_OUTPUT), "Status": "PASSED" if EXCEL_OUTPUT.exists() else "FAILED"},
    {"Export": "Figures HTML", "Path": project_relative(FIGURES_DIR), "Status": "PASSED" if len(exported_figures) == 13 else "WARNING"},
    {"Export": "Rapport qualité", "Path": project_relative(PROJECT_DIR / "exports" / "quality" / "notebook_02_quality_report.xlsx"), "Status": "à générer par scripts/quality_check_notebook_02.py"},
])
display(Export_Control)


,Export,Path,Status
0,Excel,exports\notebook_02_optimisation_resultats.xlsx,PASSED
1,Figures HTML,exports\figures\notebook_02,PASSED
2,Rapport qualité,exports\quality\notebook_02_quality_report.xlsx,à générer par scripts/quality_check_notebook_0...


## 18. Limites méthodologiques et lecture encadrant

Les résultats restent dépendants de la qualité des rendements 2025, de la revalorisation de certaines obligations, de la disponibilité des contraintes réglementaires détaillées et des hypothèses APT. Les benchmarks externes et tests de robustesse servent à documenter ces limites, sans remplacer la décision institutionnelle fondée sur le moteur interne.


## 19. Conclusion opérationnelle

Markowitz est conservé comme benchmark académique afin d'illustrer le compromis rendement-risque classique. Toutefois, la recommandation finale s'appuie sur une logique plus adaptée à une compagnie d'assurances : contrôle du risque extrême par CVaR, contrainte de turnover, diversification du risque, contribution factorielle APT, robustesse de distribution et décision multicritère.

Max Sharpe est exclu, car les faibles volatilités observées sur certaines obligations revalorisées peuvent rendre ce ratio instable. Historical_Raw n'est pas utilisé comme scénario principal. Black-Litterman n'est pas retenu faute de vues institutionnelles formalisées.

Les benchmarks externes skfolio et graph-based servent uniquement à vérifier la cohérence des résultats internes. Le portefeuille recommandé résulte ainsi d'un compromis entre rendement attendu, risque extrême, diversification, stabilité, turnover et conformité réglementaire testable.
